# Module 4: E-Commerce Customer Churn Prediction & Retention Strategy
### Senior Data Scientist & ML Engineering Portfolio
---
### Objectives:
1. **Defensible Business Churn Formulation**: Overcome the non-contractual ambiguity by instituting a 90-day future inactivity window cutoff.
2. **Leak-Free Feature Store**: Calculate historical customer features (Recency, Frequency, Monetary, Unique SKUs, Inter-Purchase Velocity) strictly before the cutoff date.
3. **Classification Benchmarking**: Train and evaluate **Logistic Regression**, **Random Forest**, and **XGBoost Classifiers** using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
4. **Prioritizing Recall in Retention**: Explain why false negatives (missing a churning VIP) are far more costly to retail EBITDA than false positives.
5. **VIP Revenue Exposure**: Identify high-value accounts at immediate risk of churning and prescribe high-touch retention workflows.


In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Out-of-Time Churn Feature Engineering
We use `src.features.churn_features.ChurnFeatureEngineer` to segment the historical timeline.
- Cutoff Date: `2011-09-10`
- Target: 1 if 0 purchases made between `2011-09-10` and `2011-12-09` (90-day window); 0 otherwise.


In [ ]:
from src.features.churn_features import ChurnFeatureEngineer

fe = ChurnFeatureEngineer(data_path=Path('../data/processed/online_retail_customer_centric.csv'), churn_window_days=90)
churn_df = fe.engineer_churn_features()
churn_df.head()


## 2. Ground-Truth Churn Distribution
Examining the natural balance between retained and churned cohorts.


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=churn_df, x='Churn', palette=['#2ca02c', '#d62728'])
plt.title('Churn Label Balance (0 = Retained, 1 = Churned)', fontsize=13, weight='bold')
plt.xticks([0, 1], ['Retained (Active in Future)', 'Churned (No Future Purchase)'])
plt.ylabel('Customer Count')
plt.show()

print(churn_df['Churn'].value_counts(normalize=True).round(3) * 100)


## 3. Training & Benchmarking Competitive Classifiers
Evaluating Logistic Regression, Random Forest, and XGBoost on an unseen 20% stratified test set.


In [ ]:
from src.models.churn_prediction import CustomerChurnPredictor

predictor = CustomerChurnPredictor(data_path=Path('../data/processed/customer_churn_features.csv'), test_size=0.20)
predictor.load_and_split()
eval_df = predictor.train_and_evaluate()
eval_df


## 4. ROC-AUC Performance Curves
Comparing True Positive Rate vs. False Positive Rate across all probability thresholds.


In [ ]:
plt.figure(figsize=(8, 6))

for name, config in predictor.models.items():
    clf = config['model']
    X_test_eval = predictor.scaler.transform(predictor.X_test) if config['is_scaled'] else predictor.X_test
    y_probs = clf.predict_proba(X_test_eval)[:, 1]
    fpr, tpr, _ = roc_curve(predictor.y_test, y_probs)
    roc_score = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_score:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title('ROC Curves - Churn Classification Benchmark', fontsize=14, weight='bold')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()


## 5. Identifying Top Churn Risk Drivers
What behavioral indicators reveal an impending customer departure?


In [ ]:
feat_imp = predictor.get_feature_importance()

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='mako')
plt.title('Predictive Feature Importance for Customer Churn', fontsize=13, weight='bold')
plt.xlabel('Normalized Gini Importance')
plt.tight_layout()
plt.show()


## 6. Business Impact: Scoring the Cohort & VIP Revenue Exposure
Translating model probabilities into high-value revenue protection.


In [ ]:
scored_df = predictor.score_all_customers()

tier_summary = scored_df.groupby('Churn_Risk_Tier').agg(
    CustomerCount=('CustomerID', 'count'),
    AvgChurnProbability=('Churn_Probability', 'mean'),
    TotalHistoricalSpend=('HistoricalMonetary', 'sum'),
    AvgHistoricalSpend=('HistoricalMonetary', 'mean')
).round(2).reset_index()

tier_summary['Revenue_Exposure (%)'] = round(
    (tier_summary['TotalHistoricalSpend'] / tier_summary['TotalHistoricalSpend'].sum()) * 100, 2
)
tier_summary


## 7. Actionable VIP Retention List
High-spending customers with >70% churn probability who require immediate executive intervention.


In [ ]:
vip_at_risk = scored_df[
    (scored_df['Churn_Risk_Tier'] == 'High Risk') & (scored_df['HistoricalMonetary'] >= 1000)
].sort_values(by='HistoricalMonetary', ascending=False)

print(f"Total High-Value Accounts at Immediate Risk: {len(vip_at_risk)}")
print(f"Total Capital Exposure: £{vip_at_risk['HistoricalMonetary'].sum():,.2f}")

vip_at_risk[['CustomerID', 'HistoricalMonetary', 'HistoricalFrequency', 'Recency', 'Churn_Probability', 'Retention_Strategy']].head(10)
